In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# STEP 1: LOAD AND PREPARE THE ACTUAL WORKBOOK
# ============================================================

FILE = Path("Problem_Set8_2026.xlsx")
if not FILE.is_file():
    raise FileNotFoundError(f"Place {FILE.name} in the notebook's working directory.")

workbook = pd.ExcelFile(FILE, engine="openpyxl")
print("Workbook sheets:", workbook.sheet_names)

raw = pd.read_excel(
    workbook, sheet_name="125 Size-Past return portfolios", header=None
)
factors = pd.read_excel(workbook, sheet_name="FF factors")
workbook.close()

# Actual layout: three adjacent blocks, each with a date and 25 portfolios.
# Excel rows 8–9 contain labels; monthly observations begin on row 10.
assert raw.shape[1] == 78
assert "simple raw monthly returns" in str(raw.iloc[5, 0]).lower()
assert "RF" in factors.columns


def monthly_index(values):
    dates = pd.to_datetime(
        pd.to_numeric(values, errors="raise").astype(int).astype(str),
        format="%Y%m",
        errors="raise",
    )
    index = pd.PeriodIndex(dates, freq="M")
    assert not index.has_duplicates, "Duplicate monthly dates."
    assert index.is_monotonic_increasing, "Dates are not sorted."
    return index


block_specs = [
    (0, "Sort1M", "Past 1-month return"),
    (26, "Sort12M", "Past 2-12-month return"),
    (52, "Sort60M", "Past 13-60-month return"),
]

blocks = []
missing_count = 0

for date_col, prefix, expected_label in block_specs:
    assert str(raw.iloc[8, date_col]).strip() == expected_label

    dates = monthly_index(raw.iloc[9:, date_col])
    block = raw.iloc[9:, date_col + 1:date_col + 26].copy()
    block = block.apply(pd.to_numeric, errors="raise")

    missing_count += int(block.eq(-999).sum().sum())

    # -999 denotes missing data. All returns are in percentage units.
    block = block.replace(-999, np.nan) / 100.0
    block.index = dates
    block.columns = [
        f"{prefix}_Size{size}_Past{return_rank}"
        for size in range(1, 6)
        for return_rank in range(1, 6)
    ]
    blocks.append(block)

assert all(block.index.equals(blocks[0].index) for block in blocks)

returns = pd.concat(blocks, axis=1)
assert returns.shape[1] == 75
assert returns.columns.is_unique

# Preserve calendar gaps so shifts refer to calendar months.
calendar = pd.period_range(returns.index.min(), returns.index.max(), freq="M")
returns = returns.reindex(calendar)
returns.index.name = "Month"

rf_dates = monthly_index(factors.iloc[:, 0])
rf_values = pd.to_numeric(factors["RF"], errors="raise")
rf = pd.Series(
    rf_values.replace(-999, np.nan).to_numpy(dtype=float) / 100.0,
    index=rf_dates,
    name="RF",
).reindex(calendar)

assert not np.isinf(returns.to_numpy()).any()
assert not np.isinf(rf.to_numpy()).any()
values = returns.to_numpy(dtype=float)
observed = values[np.isfinite(values)]

assert (observed >= -1.0).all(), (
    f"Invalid simple return below -100%: minimum = {observed.min():.6f}"
)

print("Portfolio series:", returns.shape[1])
print("Portfolio missing-value codes replaced:", missing_count)
print("Return units: percentage units converted to decimals.")
print(f"Workbook portfolio period: {calendar[0]} to {calendar[-1]}")


def construct_predictors(r):
    x1 = r.shift(1)

    # Eleven monthly returns: t-12, ..., t-2.
    x2 = (
        (1.0 + r)
        .shift(2)
        .rolling(window=11, min_periods=11)
        .apply(np.prod, raw=True)
        - 1.0
    )

    # Forty-eight monthly returns: t-60, ..., t-13.
    x3 = (
        (1.0 + r)
        .shift(13)
        .rolling(window=48, min_periods=48)
        .apply(np.prod, raw=True)
        - 1.0
    )
    return x1, x2, x3


Y = returns.sub(rf, axis=0)
X1, X2, X3 = construct_predictors(returns)

# Validate lag alignment and compounding against direct calendar slices.
assert X1.iloc[:1].isna().all().all()
assert X2.iloc[:12].isna().all().all()
assert X3.iloc[:60].isna().all().all()

for j in range(1, len(calendar)):
    t = calendar[j]
    assert calendar[j - 1] == t - 1

    np.testing.assert_allclose(
        X1.iloc[j].to_numpy(),
        returns.iloc[j - 1].to_numpy(),
        equal_nan=True,
    )

    if j >= 12:
        window2 = returns.loc[t - 12:t - 2]
        assert len(window2) == 11
        expected2 = np.prod(1.0 + window2.to_numpy(), axis=0) - 1.0
        np.testing.assert_allclose(
            X2.iloc[j].to_numpy(), expected2,
            rtol=1e-10, atol=1e-12, equal_nan=True,
        )

    if j >= 60:
        window3 = returns.loc[t - 60:t - 13]
        assert len(window3) == 48
        expected3 = np.prod(1.0 + window3.to_numpy(), axis=0) - 1.0
        np.testing.assert_allclose(
            X3.iloc[j].to_numpy(), expected3,
            rtol=1e-10, atol=1e-12, equal_nan=True,
        )

# Explicit look-ahead check: changing returns at/after t must not
# change any predictor dated t or earlier.
for cut in sorted({60, len(calendar) // 2, len(calendar) - 1}):
    altered_returns = returns.copy()
    altered_returns.iloc[cut:] = 0.12345
    altered_predictors = construct_predictors(altered_returns)

    for original, altered in zip((X1, X2, X3), altered_predictors):
        np.testing.assert_allclose(
            original.iloc[:cut + 1].to_numpy(),
            altered.iloc[:cut + 1].to_numpy(),
            rtol=1e-10, atol=1e-12, equal_nan=True,
        )

print("Validation passed: calendar lags, compounding, and no look-ahead.")

# ============================================================
# STEP 2: MONTHLY CROSS-SECTIONAL OLS
# ============================================================

coefficient_names = [
    "Intercept",
    "Past_1M",
    "Past_t12_to_t2",
    "Past_t60_to_t13",
]

coefficient_rows = []
regression_dates = []

for j, month in enumerate(calendar):
    y = Y.iloc[j].to_numpy(dtype=float)
    X = np.column_stack([
        np.ones(75),
        X1.iloc[j].to_numpy(dtype=float),
        X2.iloc[j].to_numpy(dtype=float),
        X3.iloc[j].to_numpy(dtype=float),
    ])

    # Keep only months with complete observations for all 75 portfolios.
    if not (np.isfinite(y).all() and np.isfinite(X).all()):
        continue

    if np.linalg.matrix_rank(X) != 4:
        raise ValueError(f"Regressor matrix is not full rank in {month}.")

    # Least squares gives the OLS intercept and three slopes.
    beta = np.linalg.lstsq(X, y, rcond=None)[0]

    # OLS residuals must be orthogonal to the regressors.
    residuals = y - X @ beta
    np.testing.assert_allclose(
        X.T @ residuals, np.zeros(4), rtol=0, atol=1e-8
    )

    coefficient_rows.append(beta)
    regression_dates.append(month)

monthly_coefficients = pd.DataFrame(
    coefficient_rows,
    index=pd.PeriodIndex(regression_dates, freq="M", name="Month"),
    columns=coefficient_names,
)

T = len(monthly_coefficients)
if T < 2:
    raise ValueError("At least two eligible months are needed.")

assert np.isfinite(monthly_coefficients.to_numpy()).all()

# ============================================================
# STEP 3: CONVENTIONAL FAMA–MACBETH STATISTICS
# ============================================================

average_coefficients = monthly_coefficients.mean()
coefficient_sd = monthly_coefficients.std(ddof=1)
standard_errors = coefficient_sd / np.sqrt(T)

if (standard_errors <= 0).any():
    raise ValueError("A coefficient has a zero or invalid standard error.")

summary = pd.DataFrame({
    "Average Coefficient": average_coefficients,
    "Fama–MacBeth t-statistic": average_coefficients / standard_errors,
})
summary.index.name = "Variable"

print(
    f"\nRegression sample: {monthly_coefficients.index[0]} "
    f"to {monthly_coefficients.index[-1]}"
)
print(f"Number of monthly regressions: {T:,}")
print("Portfolios per regression: 75")
display(summary.style.format({
    "Average Coefficient": "{:.6f}",
    "Fama–MacBeth t-statistic": "{:.4f}",
}))

monthly_file = Path("PS8_Qd_monthly_coefficients.csv")
summary_file = Path("PS8_Qd_Fama_MacBeth_summary.csv")

monthly_coefficients.to_csv(monthly_file, float_format="%.12g")
summary.to_csv(summary_file, float_format="%.12g")

print(f"Saved: {monthly_file.name}")
print(f"Saved: {summary_file.name}")

Workbook sheets: ['125 Size-Past return portfolios', 'FF factors']
Portfolio series: 75
Portfolio missing-value codes replaced: 1750
Return units: percentage units converted to decimals.
Workbook portfolio period: 1926-02 to 2026-06
Validation passed: calendar lags, compounding, and no look-ahead.

Regression sample: 1936-01 to 2026-06
Number of monthly regressions: 1,086
Portfolios per regression: 75


,Average Coefficient,Fama–MacBeth t-statistic
Variable,,
Intercept,0.002853,1.9096
Past_1M,0.048321,4.4414
Past_t12_to_t2,0.017010,6.2858
Past_t60_to_t13,0.002919,4.1634


Saved: PS8_Qd_monthly_coefficients.csv
Saved: PS8_Qd_Fama_MacBeth_summary.csv


The Fama–MacBeth coefficients are positive and statistically significant at all three horizons. The past one-month coefficient is 0.04832 (\(t=4.44\)), indicating short-term continuation rather than reversal. The past 12-month coefficient, skipping the latest month, is 0.01701 (\(t=6.29\)), supporting intermediate-term momentum. The past 60-month coefficient, skipping the latest year, is 0.002919 (\(t=4.16\)), indicating long-term continuation rather than reversal. Holding other predictors constant, a 10-percentage-point increase in these measures predicts approximately 48.3, 17.0, and 2.9 basis points higher monthly excess returns, respectively. The intercept is insignificant at 5% (\(t=1.91\)).

Overall, past winners tend to outperform past losers. Investor underreaction and gradual information diffusion could explain this persistence; the negative coefficients expected from overreaction followed by reversal are absent.

These results potentially challenge weak-form efficiency because historical returns predict future returns. However, predictability does not establish exploitable abnormal returns. The regressions subtract the risk-free rate but do not control for systematic risk, so the patterns could reflect risk compensation. Transaction costs, turnover, and short-selling costs could absorb trading profits. Data mining could also exaggerate significance, making risk-adjusted, out-of-sample results after costs necessary for a stronger conclusion.